**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/hernandez-perez-2020"], check=True)
    os.chdir("/content/encore/entries/hernandez-perez-2020")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Hernández-Pérez, Cooper and Newman (2020) on CRCNS hc-27

Paper: "Medial entorhinal cortex activates in a traveling wave in the rat", eLife 2020,
[doi:10.7554/elife.52289](https://doi.org/10.7554/elife.52289). Claims:
`claims/hernandez-perez-2020.yaml`, written from the version of record (PMC7046467), its Methods,
legends and Figure 1, figure supplement 1. The paper links no code, and the data set's code folder
holds a loading script only (`readData.m`, `spreadLFP.m`), so every method below is from the
paper's text, through the claims file or a passage cited where it is used.

Dataset: CRCNS hc-27, [doi:10.6080/K0C53J2R](https://doi.org/10.6080/K0C53J2R), read anonymously
from the bucket `crcnsarchive` under `hc-27/live/`, the zip opened in place over range requests.
CRCNS data sets are not versioned; the version is recorded as `unversioned`. It is the only
published state of the data set (its data descriptor is version 0.6 of February 13, 2020, and the
bucket holds no `hc-27/old/`), so it is the one pinned. The MD5 sums of the files read, from the
data set's `checksums.md5`, are printed in the data check below
(`RawSignalTravelingWaveRats.zip` c9cd6981d8559d125c02dc64c27e5b09), so a later change of the
files can be detected.

The data set holds, per rat, one matrix of LFP at 500 Hz, 5 or 6 channels ordered dorsal to
ventral, and nothing else: no units, positions, electrode geometry, behaviour, or broadband
signal.

| Claim | Tested here |
|---|---|
| fig2d-phase-shift-rate | tested |
| fig2d-linear-fit | tested |
| fig2a-all-rats | tested |
| fig2e-phase-locking-high | tested |
| fig2e-phase-locking-decrease | tested |
| fig2c-coherence | tested |
| fig2f-coherence-decrease | tested |
| fig2g-power-no-change | tested |
| fig3-along-not-across-layers | blocked, data missing (the laminar array's sites are not in the data set) |
| fig4d-mua-phase-shift | blocked, data missing (no signal above 250 Hz) |
| fig4d-mua-matches-lfp | blocked, data missing (depends on fig4d-mua-phase-shift) |
| fig5c-rise-decay-gradient | tested |
| fig5c-peak-trough-gradient | tested |
| fig5-dorsal-sawtooth | tested |
| fig5-ventral-rise-not-peak | tested |
| fig6b-delay-by-phase | tested |
| fig6c-delay-phase-slope | tested |
| fig6b-hilbert-between-limits | tested |
| fig7-asymmetry-delay | tested |
| fig7-residual-delay | tested |
| fig8c-relative-delay-flat | tested |
| fig8d-absolute-delay-slope | tested |
| fig8-correlation-strength | tested |
| fig8-coupled-oscillators | tested |

Readings. The claims file's open details (O1 to O8 in its header) and one question of units
(below) are run under every reading that can be run here:

- Units (assumption). The theta epochs need a 400 µV threshold (Methods, Data inclusion
  criteria), but the files state no unit, and the data check shows two kinds of file. Readings:
  `counts` (main): the non-integer files (rats 2 and 3) divided by 0.195 µV per bit, so that all
  rats are on the scale of the integer files, and 400 applied on that scale; `file`: 400 applied
  to the values as stored; `none`: no amplitude restriction. Under `counts` the threshold is 400
  µV if the integer files are in µV, and about 78 µV if they are Intan counts (0.195 µV per
  count). The section "Theta threshold sweep" shows the results with thresholds of 200 to 2051 on
  that scale, 2051 being 400 µV with the integer files read as Intan counts.
- O3, which channel's epochs: `a` (main) each channel's own, pairs on the times both keep; `b`
  times kept on all channels; `c` the dorsal-most channel's. Per-cycle analyses across all
  channels use `b` for readings `a` and `b`.
- O7, waveform extrema: `a` (main) local extrema of the 1-25 Hz signal; `b` one extremum between
  zero crossings of the 6-12 Hz signal (Cole and Voytek).
- O1: positions 0.566 mm times the channel index (reading a); the data set gives no positions,
  so reading b cannot be run. O2: every channel (reading a). The data set's
  author proof (`docs/MEC-TW_authorProof.pdf`, Figure 1, figure supplement 1) marks the
  electrode tips at the key recording (green) and, in blue, electrodes outside the MEC, which its
  key labels "unused electrode". Read by eye, the green tips number 6, 5, 6, 5, 6 and 5 for rats
  1 to 6, the channel counts of the files, so the files hold only the electrodes used, and
  reading b (only electrodes inside the MEC) selects the same channels as reading a. The figure
  does not say which tip is which channel, so it cannot give positions (O1 b).

In [ ]:
import time, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import fsspec
import mectw as m

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
T0 = time.time()

## Data check

In [ ]:
import warnings
md5 = fsspec.open("https://crcnsarchive.s3-us-west-2.amazonaws.com/hc-27/live/checksums.md5", "rt").open().read()
print(md5)
z = m.open_zip()
print([(i.filename, i.file_size) for i in z.infolist()])

UNITS = ["counts", "file", "none"]
SWEEP = ["counts:200", "counts:300", "counts:600", "counts:800", "counts:2051"]  # threshold sweep on the `counts` scale
CHR = ["a", "b", "c"]
EXT = ["a", "b"]
rows = []
RES = {}
CTA = {}
for r in m.RATS:
    x = m.load_rat(z, r)
    R = m.Rat(r, x)
    steps = [np.min(np.abs(d[d != 0])) for d in np.diff(x, axis=1)]
    f = np.fft.rfftfreq(5000, 1 / m.FS)
    seg = x[0, : (x.shape[1] // 5000) * 5000].reshape(-1, 5000)
    ps = (np.abs(np.fft.rfft(seg - seg.mean(1, keepdims=True), axis=1)) ** 2).mean(0)
    band = (f > 1) & (f < 30)
    line = ps[np.argmin(np.abs(f - 60))] / np.median(ps[(f > 55) & (f < 65)])
    rows.append(dict(rat=r, channels=R.nch, samples=R.n, minutes=round(R.n / m.FS / 60, 2),
                     integer_valued=R.isint, smallest_step=float(np.min(steps)),
                     min=float(x.min()), max=float(x.max()), sd_range=f"{x.std(1).min():.0f}-{x.std(1).max():.0f}",
                     n_at_32767=int((np.abs(x) >= 32767).sum()),
                     peak_1_30Hz=float(f[band][np.argmax(ps[band])]),
                     theta_peak_4_12Hz=float(f[(f > 4) & (f < 12)][np.argmax(ps[(f > 4) & (f < 12)])]),
                     line_60Hz_ratio=round(float(line), 1),
                     theta_amp_median=f"{np.median(R.amp, 1).min():.0f}-{np.median(R.amp, 1).max():.0f}",
                     kept_artefact_free=f"{R.keep.mean(1).min():.3f}-{R.keep.mean(1).max():.3f}"))
    res = {"nch": R.nch, "pos": R.pos, "isint": R.isint, "amp_med": np.median(R.amp, 1) * R.scale("counts")}
    for u in UNITS:
        ep = R.epochs(u)
        res[("epfrac", u)] = ep.mean(1)
        for c in CHR:
            res[("hil", u, c)] = m.hilbert_measures(R, u, c)
            res[("pow", u, c)] = m.theta_power(R, u, c)
    for u in SWEEP:
        res[("epfrac", u)] = R.epochs(u).mean(1)
        with warnings.catch_warnings():  # high thresholds leave some channels without epochs (NaN)
            warnings.simplefilter("ignore", RuntimeWarning)
            for c in ["a", "c"]:
                res[("hil", u, c)] = m.hilbert_measures(R, u, c)
    for w in ["a", "b"]:
        for c in ["a", "b"]:
            res[("coh", w, c)] = m.coherence_matrix(R, w, c)
    for e in EXT:
        cyc = m.all_cycles(R, e)
        res[("ncyc", e)] = [int(cy["valid"].sum()) for cy in cyc]
        for u in UNITS:
            if u == "file" and not R.epochs(u)[0].any():
                continue
            for c in CHR:
                res[("asy", u, c, e)] = m.asymmetry_by_channel(R, cyc, u, c)
            for c in ["b", "c"]:
                res[("del", u, c, e)] = m.delays(R, cyc, u, c)
        if e == "a":
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", RuntimeWarning)
                for u in SWEEP:
                    res[("asy", u, "a", e)] = m.asymmetry_by_channel(R, cyc, u, "a")
                    for c in ["b", "c"]:
                        res[("del", u, c, e)] = m.delays(R, cyc, u, c)
            # cycle-triggered average of the raw LFP on the dorsal-most channel's peaks
            # (Methods, Cycle-triggered averages theta waves), theta epochs of the dorsal-most channel
            ep0 = R.epochs("counts")[0]
            pk = cyc[0]["peak"][cyc[0]["valid"] & ep0[cyc[0]["peak"]]]
            pk = pk[(pk > 125) & (pk < R.n - 125)]
            # whole window artefact-free on every channel
            cb = np.concatenate([[0], np.cumsum(~R.keep.all(0))])
            pk = pk[cb[pk + 63] - cb[pk - 62] == 0]
            idx = pk[:, None] + np.arange(-62, 63)[None, :]
            CTA[r] = np.array([R.x[c][idx].mean(0) for c in range(R.nch)])
    RES[r] = res
    del R, x
data = pd.DataFrame(rows)
print(data.to_string())
print(f"elapsed {time.time() - T0:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

In [ ]:
# theta epochs and cycle counts under each unit reading
ep_rows = []
for r in m.RATS:
    res = RES[r]
    row = dict(rat=r)
    for u in UNITS:
        row[f"epoch fraction {u}"] = " ".join(f"{v:.2f}" for v in res[("epfrac", u)])
        d = res.get(("del", u, "c", "a"))
        row[f"dorsal cycles {u}"] = d["n_ref"] if d else 0
        d = res.get(("del", u, "b", "a"))
        row[f"all-channel cycles {u}"] = d["n_ref"] if d else 0
    row["valid cycles, whole recording (O7 a)"] = res[("ncyc", "a")][0]
    ep_rows.append(row)
print(pd.DataFrame(ep_rows).to_string())
print("channels per rat:", {r: RES[r]["nch"] for r in m.RATS})

# median theta (6-12 Hz) amplitude per channel on the `counts` scale, and in µV under the two
# readings of the integer files, beside the paper's 400 µV threshold
amp_rows = []
for r in m.RATS:
    a = RES[r]["amp_med"]
    amp_rows.append(dict(rat=r, integer_file=RES[r]["isint"],
                         common_scale=" ".join(f"{v:.0f}" for v in a),
                         uV_if_integer_files_are_uV=" ".join(f"{v:.0f}" for v in a),
                         uV_if_integer_files_are_Intan_counts=" ".join(f"{v * m.INTAN_UV_PER_BIT:.0f}" for v in a)))
print(pd.DataFrame(amp_rows).to_string())
print("threshold 400 µV on the common scale: 400 if the integer files are in µV, "
      f"{400 / m.INTAN_UV_PER_BIT:.0f} if they are Intan counts; the `counts` reading applies 400")

In [ ]:
# duplicate recordings: compare the first 20 s of every channel of every rat
zz = m.open_zip()
heads = {}
for r in m.RATS:
    x = m.load_rat(zz, r)
    for c in range(x.shape[0]):
        heads[(r, c)] = x[c, :10000] - x[c, :10000].mean()
keys = list(heads)
maxc = 0
pair = None
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        if keys[i][0] == keys[j][0]:
            continue
        cc = np.corrcoef(heads[keys[i]], heads[keys[j]])[0, 1]
        if abs(cc) > maxc:
            maxc, pair = abs(cc), (keys[i], keys[j])
same_within = [(r, a, b) for r in m.RATS for a in range(RES[r]["nch"]) for b in range(a + 1, RES[r]["nch"])
               if np.array_equal(heads[(r, a)], heads[(r, b)])]
print(f"largest |r| between channels of different rats over the first 20 s: {maxc:.3f} {pair}")
print("identical channels within a rat:", same_within)

What the data check shows, against the descriptor and the paper:

- Six rats, as the descriptor and the paper (Methods, Animals) say; 6, 5, 6, 5, 6 and 5 channels.
  No channel repeats another, within or across rats (the largest correlation between rats is
  printed above).
- Sampling rate: the descriptor's 500 Hz is consistent with the data: the spectra peak at
  theta (6.7 to 7.4 Hz in the 4-12 Hz band) and line noise stands out at 60 Hz in every rat. This
  is consistent with the stated rate, not a proof of it.
- Duration: 20.1 to 33.5 min; the paper's trials last 25 min (Behavioral protocol), so rat 5
  holds more than one trial's length, and the others less.
- Units: none is stated. Rats 1, 4, 5 and 6 hold integers (smallest step 1) and rat 5 reaches
  32767, the int16 limit: these look like ADC counts. Rats 2 and 3 hold non-integers about five
  times smaller in spread. With the Intan step of 0.195 µV, the theta amplitudes of all rats are
  on one scale only if rats 2 and 3 are in µV and the others in counts. Under the paper's 400 µV
  threshold applied as stored (`file`), rats 2 and 3 have no theta epoch at all, so no test
  over six rats is possible; under `counts` all rats have epochs, and the dorsal-most channel of
  rats 2 to 6 gives 3200 to 6286 cycles, near the paper's "Between 3500 to 5000 individual theta
  cycles ... for each phase in each rat" (Methods, Conduction delays). Rat 1's dorsal-most channel
  has low theta amplitude and gives 815. The main reading is `counts`, as an assumption; `none` is
  shown throughout. The second table above gives each channel's median theta amplitude: on the
  common scale 268 to 1029, which is 52 to 201 µV if the integer files are Intan counts. Under
  that reading of the integer files the paper's 400 µV is 2051 on the common scale, above every
  channel's median, and the sweep below shows it leaves no theta epoch in any rat. So the
  `counts` reading, which applies 400 to the integer files as stored, puts the threshold at 400 µV
  only if those files are in µV; if they are Intan counts it is about 78 µV.
- Clock: each rat is one matrix whose channels share one sample clock; the data set holds no
  trial table, behaviour or annotation, so no check against task events is possible.
- Positions and channel selection: the data set gives no positions; they are taken from the
  nominal 0.566 mm spacing (O1 a). Every channel is used (O2 a); the data set's author proof
  (Figure 1, figure supplement 1) marks as many electrodes at the key recording as the files hold
  channels, and labels the electrodes outside the MEC "unused" (see the header), so O2 b selects
  the same channels.

### Helper check on synthetic data

Five channels of an asymmetric 8 Hz wave (fundamental and second harmonic, fast rise and slow
decay) delayed by 5 ms per channel
(8.83 ms/mm at 0.566 mm), with noise: the Hilbert slope should be 8.83 ms/mm times 8 Hz times 360,
25.4 deg/mm, every phase's conduction delay 8.83 ms/mm, the asymmetry-related delay 0, and the
rise/decay index negative and equal on all channels.

In [ ]:
rng_syn = np.random.default_rng(7)
tt = np.arange(int(300 * m.FS)) / m.FS
def wave(t):  # fundamental and second harmonic, both inside the 1-25 Hz band, so filtering keeps the shape
    return 1000 * (np.sin(2 * np.pi * 8 * t) + 0.3 * np.sin(2 * np.pi * 16 * t))
fine = np.arange(0, 0.25, 1e-6)
wf = wave(fine)
pk_f = fine[np.flatnonzero((wf[1:-1] > wf[:-2]) & (wf[1:-1] >= wf[2:])) + 1]
tr_f = fine[np.flatnonzero((wf[1:-1] < wf[:-2]) & (wf[1:-1] <= wf[2:])) + 1]
rise_true = pk_f[1] - tr_f[tr_f < pk_f[1]][-1]
decay_true = tr_f[tr_f > pk_f[1]][0] - pk_f[1]
print(f"noise-free wave: rise {rise_true * 1000:.2f} ms, decay {decay_true * 1000:.2f} ms, index {np.log10(rise_true / decay_true):.3f}")
xs = np.array([wave(tt - 0.005 * c) + rng_syn.normal(0, 50, tt.size) for c in range(5)])
S = m.Rat(0, xs)
hs = m.hilbert_measures(S, "none", "a")
print("Hilbert offsets (deg):", np.round(hs["offset"], 1), "slope", round(m.lsq_slope(S.pos, hs["offset"]), 2), "deg/mm (expected 25.4)")
cs = m.all_cycles(S, "a")
ds = m.delays(S, cs, "none", "b")
print("delays (ms/mm):", {ph: round(float(np.median(ds[ph])), 2) for ph in m.PHASES}, "(expected 8.83)")
print("asymmetry-related delay:", round(float(np.median(ds["asym_cycle"])), 3), "(expected 0)")
print("rise/decay index:", np.round(m.asymmetry_by_channel(S, cs, "none", "a")["rise_decay"], 3), f"(expected {np.log10(rise_true / decay_true):.3f}, within the 2 ms sampling step)")
del S

In [ ]:
def per_rat(fn):
    return np.array([fn(r) for r in m.RATS], float)

def show_test(v, label):
    tp, p = m.signrank(v)
    print(f"{label}: values {np.round(v, 3)}, signed rank T+ = {tp:.0f} of 21, exact p = {p:.4f}")
    return tp, p

## Theta threshold sweep

The `counts` reading applies 400 on the scale of the integer files, so the threshold is 400 µV
only if those files are in µV (see the data check). Here the main measures are recomputed with
thresholds of 200 to 2051 on that scale (2051 is 400 µV if the integer files are Intan counts),
beside the number of reference cycles per rat, which the paper gives as "Between 3500 to 5000
individual theta cycles ... for each phase in each rat" (Methods, Conduction delays of specific
theta phases). Main readings otherwise (O7 a; fig2d and fig2e on O3 a and c; delays on O3 b and
c). NaN: some rat has no epoch or no reference cycle at that threshold.


In [ ]:
rows = []
for u in ["counts:200", "counts:300", "counts", "counts:600", "counts:800", "counts:2051"]:
    thr = 400 if u == "counts" else int(u.split(":")[1])
    nd = [RES[r][("del", u, "c", "a")]["n_ref"] for r in m.RATS]
    nbc = [RES[r][("del", u, "b", "a")]["n_ref"] for r in m.RATS]
    row = dict(threshold=thr, dorsal_epochs_per_rat=" ".join(f"{RES[r][('epfrac', u)][0]:.2f}" for r in m.RATS),
               cycles_O3c=" ".join(map(str, nd)), cycles_O3b=" ".join(map(str, nbc)),
               rats_3500_5000_O3c=sum(3500 <= v <= 5000 for v in nd))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        def hok(c):
            return not any(np.isnan(RES[r][("hil", u, c)]["offset"]).any() for r in m.RATS)
        row["fig2d_mean_slope_a"] = per_rat(lambda r: m.lsq_slope(RES[r]["pos"], RES[r][("hil", u, "a")]["offset"])).mean() if hok("a") else np.nan
        for c in ["a", "c"]:
            row[f"fig2e_mrl_{c}"] = np.median(per_rat(lambda r: np.median(RES[r][("hil", u, c)]["mrl"][1:]))) if hok(c) else np.nan
            row[f"fig2e_slope_{c}"] = per_rat(lambda r: m.lsq_slope(RES[r]["pos"], RES[r][("hil", u, c)]["mrl"])).mean() if hok(c) else np.nan
        rd0 = per_rat(lambda r: RES[r][("asy", u, "a", "a")]["rise_decay"][0])
        rd1 = per_rat(lambda r: RES[r][("asy", u, "a", "a")]["rise_decay"][-1])
        row["fig5c_rd_dorsal"], row["fig5c_rd_ventral"] = np.median(rd0), np.median(rd1)
        for c in ["b", "c"]:
            D = {r: RES[r][("del", u, c, "a")] for r in m.RATS}
            ok = all(D[r]["n_ref"] > 0 for r in m.RATS)
            for k, key in [("fall", "fall"), ("peak", "peak"), ("rise", "rise_mid"), ("asym", "asym_cycle")]:
                row[f"{k}_{c}"] = np.median(per_rat(lambda r: np.median(D[r][key]))) if ok else np.nan
    rows.append(row)
sweep = pd.DataFrame(rows)
print(sweep.set_index("threshold").T.to_string())
print("paper: fig2e MRL 0.90 [0.85-0.95], slope -0.067 +/- 0.011; fig5c -0.43 and -0.11; fall 5.47, peak 12.00 ms/mm; asymmetry delay 3.56 [1.56-5.89] ms/mm")


What the sweep shows. At 400 µV with the integer files read as Intan counts (2051) no rat has a
theta epoch, so that reading cannot be run. No threshold from 200 to 2051 gives 3500 to 5000
reference cycles in every rat: on the dorsal-most channel's epochs, 400 brings two rats into that
range (rats 3 and 5; rat 1 has 815), 200 one, 300 one, and 600 one. The measures move with the
threshold: the median phase locking against the dorsal-most electrode is 0.87 to 0.97 (O3 c) and
its slope -0.113 to -0.048 per mm; the asymmetry-related delay is 6.0 to 7.6 ms/mm at every
threshold where it can be computed, never within the paper's 3.56 +/- 2.33; the dorsal rise/decay
index is -0.14 to -0.18 against the paper's -0.43. The paper's cycle counts do not single out a
threshold, so the outcomes that depend on the epochs stay marked as resting on the assumption.


## fig2d-phase-shift-rate

> Dorsal MEC sites are phase advanced relative to ventral sites, the theta phase shifting at
> 26.36 +/- 2.64 deg/mm along the dorsal-ventral axis, a shift significantly different from zero
> across six rats.

Per rat, the Hilbert phase offset of each channel against the dorsal-most (circular mean of the
phase difference over the theta epochs, positive when the ventral site lags), and the slope of
offset on distance (O6 a: least squares with intercept; b: through the origin). The centre is the
mean (O6 a) or median (b) of the six slopes. Test: exact two-sided signed-rank test of the six
slopes against zero.

In [ ]:
rows = []
for u in UNITS:
    for c in CHR:
        if any(np.isnan(RES[r][("hil", u, c)]["offset"]).any() for r in m.RATS):
            rows.append(dict(units=u, O3=c, note="no theta epochs in some rat"))
            continue
        for sl in ["a", "b"]:
            s = per_rat(lambda r: m.lsq_slope(RES[r]["pos"], RES[r][("hil", u, c)]["offset"], intercept=(sl == "a")))
            tp, p = m.signrank(s)
            rows.append(dict(units=u, O3=c, slope=sl, per_rat=" ".join(f"{v:.1f}" for v in s),
                             mean=s.mean(), median=np.median(s), sem=s.std(ddof=1) / np.sqrt(6),
                             all_positive=bool((s > 0).all()), T_plus=tp, p=p))
fig2d = pd.DataFrame(rows)
print(fig2d.round(4).to_string())
main = fig2d[(fig2d.units == "counts") & (fig2d.O3 == "a") & (fig2d.slope == "a")].iloc[0]
in_tol = fig2d[(fig2d["mean"].sub(26.36).abs() <= 2.64) | (fig2d["median"].sub(26.36).abs() <= 2.64)]
print(f"main reading (counts, O3 a, slope a): mean {main['mean']:.2f} deg/mm (SEM {main['sem']:.2f}), median {main['median']:.2f}, p = {main['p']:.4f}")
print(f"combinations with mean or median within 26.36 +/- 2.64 and p <= 0.0625: {int(((in_tol.p <= 0.0625) & in_tol.all_positive).sum())} of {fig2d['p'].notna().sum()}")
for r in m.RATS:
    print(f"rat {r} offsets (counts, a):", np.round(RES[r][('hil', 'counts', 'a')]['offset'], 1))

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
for r in m.RATS:
    ax.plot(RES[r]["pos"], RES[r][("hil", "counts", "a")]["offset"], "o-", label=f"rat {r}")
ax.set_xlabel("distance from dorsal-most electrode (mm)")
ax.set_ylabel("theta phase offset (deg)")
ax.legend(fontsize=7)
allo = np.concatenate([RES[r][("hil", "counts", "a")]["offset"] for r in m.RATS])
print(f"plotted offsets range {allo.min():.1f} to {allo.max():.1f} deg; none outside the axes")

**Result: reproduced.** Under the main reading the six per-rat slopes average 27.32 deg/mm (SEM
3.27; median 27.04), within 26.36 +/- 2.64, all six positive (ventral sites lag; 17.1 to 38.0
deg/mm), signed-rank T+ = 21 of 21, p = 0.031. All 12 combinations of the unit, O3 and O6 readings
that can be run give a mean or median within tolerance with the same test result (means 25.6 to
27.7), so the outcome does not rest on the unit reading; under `file` rats 2 and 3 have no theta
epoch and no test over six rats can be made.

## fig2d-linear-fit

> The phase offsets are fit well by a linear relation with distance (r2 = 0.7627).

Readings: (a) squared Pearson r over all rats' points pooled, dorsal-most (0, 0) included; (b)
without the dorsal-most points; (c) the median of per-rat r2.

In [ ]:
rows = []
for u in ["counts", "none"]:
    for c in CHR:
        P = np.concatenate([RES[r]["pos"] for r in m.RATS])
        O = np.concatenate([RES[r][("hil", u, c)]["offset"] for r in m.RATS])
        nz = P > 0
        ra = stats.pearsonr(P, O)[0] ** 2
        rb = stats.pearsonr(P[nz], O[nz])[0] ** 2
        rc = np.median([stats.pearsonr(RES[r]["pos"], RES[r][("hil", u, c)]["offset"])[0] ** 2 for r in m.RATS])
        rows.append(dict(units=u, O3=c, r2_pooled=ra, r2_pooled_no_dorsal=rb, r2_median_per_rat=rc))
fig2dl = pd.DataFrame(rows)
print(fig2dl.round(4).to_string())
print("within 0.7627 +/- 0.0763:")
print((fig2dl[["r2_pooled", "r2_pooled_no_dorsal", "r2_median_per_rat"]].sub(0.7627).abs() <= 0.0763).to_string())

**Result: reproduced.** The pooled r2 with the dorsal-most points (reading a) is 0.778 under the
main reading, within 0.7627 +/- 0.0763, and 0.765 to 0.778 under every unit and O3 reading shown.
Without the dorsal-most points (b) it is 0.66 to 0.68, and the median per rat (c) is 0.95; neither
is within tolerance, and the criterion holds when one reading is.

## fig2a-all-rats

> Gradual theta phase shifts along the long axis of the MEC are seen in every one of the six rats.

Per rat, the Hilbert slope (O6 a) and the cycle-triggered average of the raw LFP on each channel,
triggered on the waveform-method peaks of the dorsal-most channel in its theta epochs (Methods,
Cycle-triggered averages), over +/-124 ms; the lag of each channel's peak is the local maximum of
its average above the average's mean nearest the trigger (standard choice; small bumps inside a
trough are not peaks).

In [ ]:
lag = {}
for r in m.RATS:
    a = CTA[r]
    t = np.arange(-62, 63) / m.FS * 1000
    # each channel's local maximum above its mean, nearest zero lag (one theta cycle spans the +/-124 ms window)
    lg = []
    for c in range(a.shape[0]):
        pk = np.flatnonzero((a[c, 1:-1] > a[c, :-2]) & (a[c, 1:-1] >= a[c, 2:]) & (a[c, 1:-1] > a[c].mean())) + 1
        lg.append(t[pk[np.argmin(np.abs(t[pk]))]])
    lag[r] = np.array(lg)
s = per_rat(lambda r: m.lsq_slope(RES[r]["pos"], RES[r][("hil", "counts", "a")]["offset"]))
tab = pd.DataFrame(dict(rat=m.RATS, hilbert_slope=s,
                        cta_peak_lag_ms=[" ".join(f"{v:.0f}" for v in lag[r]) for r in m.RATS],
                        ventral_after_dorsal=[lag[r][-1] > lag[r][0] for r in m.RATS]))
print(tab.round(2).to_string())
print("rats with positive slope and ventral CTA peak after dorsal:", int(((s > 0) & tab.ventral_after_dorsal).sum()), "of 6")
fig, axs = plt.subplots(1, 6, figsize=(14, 3), sharey=False)
for ax, r in zip(axs, m.RATS):
    a = CTA[r]
    sc = np.abs(a).max()
    for c in range(a.shape[0]):
        ax.plot(np.arange(-62, 63) * 2, a[c] / sc - c, "k")
    ax.set_title(f"rat {r}")
    ax.set_xlabel("ms from dorsal peak")

**Result: reproduced.** All six rats have a positive slope (17.1 to 38.0 deg/mm), and in each the
peak of the cycle-triggered average on the ventral-most channel follows that of the dorsal-most
(lags printed above, 16 to 40 ms on the ventral-most channel). Triggers are kept only where the
whole window is artefact-free on every channel.


## fig2e-phase-locking-high

> The phase offset between sites is stable over the recording, the median phase locking between
> sites being 0.90.

Mean resultant length of the Hilbert phase differences over theta epochs. Readings: (a) each
electrode against the dorsal-most, median per rat then across rats; (b) all pairs, likewise; (c)
all values of all rats pooled (the against-dorsal values).

In [ ]:
rng = np.random.default_rng(2051)
rows = []
for u in UNITS:
    for c in CHR:
        if any(np.isnan(RES[r][("hil", u, c)]["mrl"]).any() for r in m.RATS):
            continue
        A = per_rat(lambda r: np.median(RES[r][("hil", u, c)]["mrl"][1:]))
        B = per_rat(lambda r: np.nanmedian(RES[r][("hil", u, c)]["pair_mrl"]))
        C = np.concatenate([RES[r][("hil", u, c)]["mrl"][1:] for r in m.RATS])
        rows.append(dict(units=u, O3=c, a_vs_dorsal=np.median(A), b_all_pairs=np.median(B), c_pooled=np.median(C),
                         a_per_rat=" ".join(f"{v:.2f}" for v in A)))
fig2e = pd.DataFrame(rows)
print(fig2e.round(3).to_string())
A = per_rat(lambda r: np.median(RES[r][("hil", "counts", "a")]["mrl"][1:]))
print("main reading (a) bootstrap 95% CI of the median over rats:", np.round(m.boot_ci(A, rng), 3))

**Result: reproduced, resting on the unit assumption.** With the dorsal-most channel's theta epochs
(O3 c) the median phase locking against the dorsal-most electrode is 0.946 and over all pairs 0.936,
inside 0.85 to 0.95. With each channel's own epochs (O3 a, the main reading) or all channels' (b) it
is 0.959 to 0.971, above the range, and with no amplitude restriction (`none`) 0.78 to 0.81, below
it. The criterion holds when one reading is inside the range; which one is depends on the epochs
used. In the threshold sweep the
median against the dorsal-most electrode (O3 c) is 0.87 at 200, 0.92 at 300, 0.95 at 400 and 0.97
at 600 on the common scale, so it is inside the range for thresholds of 200 to 400.


## fig2e-phase-locking-decrease

> Phase locking decreases modestly but reliably with distance between electrodes, at -0.067 per
> mm.

Per rat, the least-squares slope of the MRL against the dorsal-most electrode on distance, with or
without the dorsal-most's own 1; centre the mean or the median over rats.

In [ ]:
rows = []
for u in UNITS:
    for c in CHR:
        if any(np.isnan(RES[r][("hil", u, c)]["mrl"]).any() for r in m.RATS):
            continue
        for incl in [True, False]:
            s = per_rat(lambda r: m.lsq_slope(RES[r]["pos"][0 if incl else 1:], RES[r][("hil", u, c)]["mrl"][0 if incl else 1:]))
            tp, p = m.signrank(s)
            rows.append(dict(units=u, O3=c, with_dorsal=incl, per_rat=" ".join(f"{v:.3f}" for v in s),
                             mean=s.mean(), median=np.median(s), n_negative=int((s < 0).sum()), T_plus=tp, p=p))
fig2e2 = pd.DataFrame(rows)
print(fig2e2.round(4).to_string())
ok = ((fig2e2["mean"] + 0.067).abs() <= 0.011) | ((fig2e2["median"] + 0.067).abs() <= 0.011)
print("combinations within -0.067 +/- 0.011 with p <= 0.0625:", int((ok & (fig2e2.p <= 0.0625)).sum()), "of", len(fig2e2))

**Result: reproduced, resting on the unit assumption.** Phase locking falls with distance in all six
rats (T+ = 0, p = 0.031) under every reading. With the dorsal-most channel's epochs (O3 c) the mean
slope is -0.0675 per mm and the median -0.0619 (dorsal-most value included), within -0.067 +/-
0.011. With each channel's own epochs (main reading) the slope is shallower (mean -0.039, median
-0.030) and with no amplitude restriction steeper (-0.13 to -0.14), both outside the tolerance. In the threshold sweep the
mean slope (O3 c) is -0.113, -0.086, -0.067 and -0.048 per mm at 200, 300, 400 and 600; only 400
is within tolerance under O3 c (with O3 a, 300 gives -0.060).


## fig2c-coherence

> Theta coherence across the MEC is 0.73.

Coherence averaged over 6, 7, ... 12 Hz on artefact-free data (not restricted to theta epochs).
Windows: (a) 1 s Hann, 50% overlap; (b) cohere's default, 256 samples, no overlap. Pairs: against
the dorsal-most, or all pairs; median per rat, then across rats. Artefact mask: each pair's own
(`a`) or all channels' (`b`).

In [ ]:
rng = np.random.default_rng(2052)
rows = []
for w in ["a", "b"]:
    for c in ["a", "b"]:
        D = per_rat(lambda r: np.median(RES[r][("coh", w, c)][0, 1:]))
        A = per_rat(lambda r: np.nanmedian(RES[r][("coh", w, c)]))
        rows.append(dict(window=w, mask=c, vs_dorsal=np.median(D), all_pairs=np.median(A),
                         vs_dorsal_per_rat=" ".join(f"{v:.2f}" for v in D), all_pairs_per_rat=" ".join(f"{v:.2f}" for v in A)))
fig2c = pd.DataFrame(rows)
print(fig2c.round(3).to_string())
D = per_rat(lambda r: np.median(RES[r][("coh", "a", "a")][0, 1:]))
print("window a, against dorsal: bootstrap 95% CI", np.round(m.boot_ci(D, rng), 3))
print("inside 0.66-0.81:")
print(((fig2c[["vs_dorsal", "all_pairs"]] >= 0.66) & (fig2c[["vs_dorsal", "all_pairs"]] <= 0.81)).to_string())

**Result: reproduced.** Over all electrode pairs the median coherence is 0.677 to 0.690 under the
four window and mask readings, inside 0.66 to 0.81. Against the dorsal-most electrode alone it is
0.605 to 0.621, below the range. The criterion holds when one combination is inside the range.

## fig2f-coherence-decrease

> Theta coherence decreases with distance between electrode pairs, at 0.067 per mm.

Per rat, the least-squares slope of coherence between each electrode and the dorsal-most on
distance (the dorsal-most itself has no coherence value and is left out).

In [ ]:
rows = []
for w in ["a", "b"]:
    for c in ["a", "b"]:
        s = per_rat(lambda r: m.lsq_slope(RES[r]["pos"][1:], RES[r][("coh", w, c)][0, 1:]))
        tp, p = m.signrank(s)
        rows.append(dict(window=w, mask=c, per_rat=" ".join(f"{v:.3f}" for v in s), mean=s.mean(),
                         median=np.median(s), n_negative=int((s < 0).sum()), T_plus=tp, p=p))
fig2f = pd.DataFrame(rows)
print(fig2f.round(4).to_string())
ok = ((fig2f["mean"].abs() - 0.067).abs() <= 0.018) | ((fig2f["median"].abs() - 0.067).abs() <= 0.018)
print("combinations with |mean| or |median| within 0.067 +/- 0.018:", int(ok.sum()), "of", len(fig2f))

**Result: not reproduced.** Coherence falls with distance in all six rats (T+ = 0, p = 0.031), but
about three times as fast as the paper's rate: the mean slope is -0.211 to -0.220 per mm and the
median -0.222 to -0.224, none within 0.067 +/- 0.018.

## fig2g-power-no-change

> Theta power does not change significantly along the long axis of the MEC.

A claim of absence, judged on the exact signed-rank test of the six per-rat slopes of theta power
on distance; reproduced when p > 0.0625. Relative power (each rat's across-electrode mean set to
1) is the test; raw power is a variant. Power from Welch or from the squared Hilbert amplitude,
under the O3 and unit readings.

In [ ]:
rng = np.random.default_rng(2053)
rows = []
for u in UNITS:
    for c in CHR:
        for meas in ["welch", "hilbert"]:
            vals = {r: RES[r][("pow", u, c)][meas] for r in m.RATS}
            if any(np.isnan(v).any() for v in vals.values()):
                continue
            for rel in [True, False]:
                s = per_rat(lambda r: m.lsq_slope(RES[r]["pos"], vals[r] / vals[r].mean() if rel else vals[r]))
                tp, p = m.signrank(s)
                rows.append(dict(units=u, O3=c, measure=meas, relative=rel, median=np.median(s),
                                 per_rat=" ".join(f"{v:.3g}" for v in s), T_plus=tp, p=p, absent=p > 0.0625))
fig2g = pd.DataFrame(rows)
print(fig2g.round(4).to_string())
for meas in ["welch", "hilbert"]:
    vals = {r: RES[r][("pow", "counts", "a")][meas] for r in m.RATS}
    s = per_rat(lambda r: m.lsq_slope(RES[r]["pos"], vals[r] / vals[r].mean()))
    print(f"main ({meas}, relative, counts, O3 a): median slope {np.median(s):.3f} /mm, 95% CI {np.round(m.boot_ci(s, rng), 3)}")
rel = fig2g[fig2g.relative]
print("relative-power combinations with p > 0.0625:", int(rel.absent.sum()), "of", len(rel))

**Result: reproduced.** The paper's test does not find a change of relative theta power with
distance under any of the 12 readings (p = 0.094 to 0.219). Under
the main reading the median slope of relative power is -0.098 per mm with Welch power (95% CI
-0.221 to 0.037) and -0.107 with Hilbert power; the paper's median is +0.73 AU/mm. Raw power gives
the same test result (p = 0.156 to 0.313).

## fig5c-rise-decay-gradient

> The rise/decay asymmetry of theta is strongly negative at dorsal sites and significantly less so
> at ventral sites (-0.43 against -0.11).

Per cycle log10(rise/decay) on the 1-25 Hz signal; per channel the median over valid cycles in the
theta epochs; across rats the median. Sites: (a) dorsal-most against ventral-most; (b) dorsal-most
against the channel nearest 2.8 mm. With 0.566 mm spacing the channel nearest 2.8 mm is the
ventral-most in every rat (2.83 mm with six channels; 2.26 mm, the last, with five), so (a) and (b)
are the same.

In [ ]:
def asy_table(key):
    rows = []
    for u in UNITS:
        for c in CHR:
            for e in EXT:
                if not all(("asy", u, c, e) in RES[r] for r in m.RATS):
                    continue
                d = per_rat(lambda r: RES[r][("asy", u, c, e)][key][0])
                v = per_rat(lambda r: RES[r][("asy", u, c, e)][key][-1])
                tp, p = m.signrank(v - d)
                rows.append(dict(units=u, O3=c, O7=e, dorsal=np.median(d), ventral=np.median(v),
                                 dorsal_per_rat=" ".join(f"{x:.2f}" for x in d),
                                 ventral_per_rat=" ".join(f"{x:.2f}" for x in v), T_plus_ventral_minus_dorsal=tp, p=p))
    return pd.DataFrame(rows)

rng = np.random.default_rng(2054)
fig5rd = asy_table("rise_decay")
print(fig5rd.round(4).to_string())
ok = ((fig5rd.dorsal + 0.43).abs() <= 0.14) & ((fig5rd.ventral + 0.11).abs() <= 0.02)
print("combinations with both medians in tolerance:", int(ok.sum()), "of", len(fig5rd))
print("cycles per channel (counts, a, O7 a):", {r: RES[r][("asy", "counts", "a", "a")]["n"].tolist() for r in m.RATS})
# variant: natural log (the Figure 5A legend says "log", the Methods log10); the per-channel and
# across-rat medians commute with the factor ln 10, so this is exact
LN10 = np.log(10)
ln = fig5rd[["units", "O3", "O7"]].assign(dorsal_ln=fig5rd.dorsal * LN10, ventral_ln=fig5rd.ventral * LN10)
okln = ((ln.dorsal_ln + 0.43).abs() <= 0.14) & ((ln.ventral_ln + 0.11).abs() <= 0.02) & (fig5rd.p <= 0.0625) & (fig5rd.ventral > fig5rd.dorsal)
print(ln.assign(meets_every_part=okln).round(3).to_string())
print("natural-log variant, combinations meeting every part:", int(okln.sum()), "of", len(ln))


**Result: partially reproduced.** Dorsal sites are more asymmetric than ventral ones in every rat
under the main reading (T+ = 21 for ventral minus dorsal, p = 0.031; p = 0.031 or 0.0625 under the
other readings), but both values differ from the paper's: the dorsal median is -0.171 against -0.43
+/- 0.14 and the ventral-most -0.049 against -0.11 +/- 0.02 under the main reading, and no reading
brings both within tolerance (dorsal -0.136 to -0.176). The gradient is present but weaker. The
criterion defines only "reproduced"; this is recorded as partially reproduced, as the direction and
the test hold and the values do not (the verifier would record it as not reproduced).

Natural-log variant: the Methods say log10, the Figure 5A legend only "log". With the natural log
the main reading gives -0.393 and -0.112, both within tolerance, and 5 of 12 combinations meet every
part. The claims file has log10; whether it should allow the natural log is asked of a reader
(`questions/hernandez-perez-2020.yaml`), and the outcome is that of the claims file's log10.


## fig5c-peak-trough-gradient

> The peak/trough asymmetry of theta is positive at dorsal sites and significantly lower at
> ventral sites (0.21 against -0.10).

In [ ]:
fig5pt = asy_table("peak_trough")
print(fig5pt.round(4).to_string())
ok = ((fig5pt.dorsal - 0.21).abs() <= 0.21) & ((fig5pt.ventral + 0.10).abs() <= 0.11) & (fig5pt.p <= 0.0625) & (fig5pt.T_plus_ventral_minus_dorsal == 0)
print("combinations with both medians in tolerance, ventral lower in all rats, p <= 0.0625:", int(ok.sum()), "of", len(fig5pt))
lnpt = fig5pt[["units", "O3", "O7"]].assign(dorsal_ln=fig5pt.dorsal * LN10, ventral_ln=fig5pt.ventral * LN10)
okln = ((lnpt.dorsal_ln - 0.21).abs() <= 0.21) & ((lnpt.ventral_ln + 0.10).abs() <= 0.11) & (fig5pt.p <= 0.0625) & (fig5pt.T_plus_ventral_minus_dorsal == 0)
print("natural-log variant: dorsal", np.round(lnpt.dorsal_ln.iloc[0], 3), "ventral", np.round(lnpt.ventral_ln.iloc[0], 3),
      "(main); combinations meeting every part:", int(okln.sum()), "of", len(lnpt))


**Result: reproduced.** Under the main reading the dorsal peak/trough index has a median of 0.110
(within 0.21 +/- 0.21) and the ventral-most -0.038 (within -0.10 +/- 0.11), lower in every rat (T+ =
0, p = 0.031). All 12 readings meet every part. With the natural log (variant,
as for fig5c-rise-decay-gradient) the main reading gives 0.253 and -0.088, and all 12 readings
still meet every part.


## fig5-dorsal-sawtooth

> At dorsal sites the rising phase of theta is significantly shorter than the decaying phase and
> the peaks are longer than the troughs.

In [ ]:
rows = []
for u in UNITS:
    for c in CHR:
        for e in EXT:
            if not all(("asy", u, c, e) in RES[r] for r in m.RATS):
                continue
            rd = per_rat(lambda r: RES[r][("asy", u, c, e)]["rise_decay"][0])
            pt = per_rat(lambda r: RES[r][("asy", u, c, e)]["peak_trough"][0])
            t1, p1 = m.signrank(rd)
            t2, p2 = m.signrank(pt)
            rows.append(dict(units=u, O3=c, O7=e, rise_decay=" ".join(f"{x:.2f}" for x in rd), T_rd=t1, p_rd=p1,
                             peak_trough=" ".join(f"{x:.2f}" for x in pt), T_pt=t2, p_pt=p2,
                             rd_holds=(p1 <= 0.0625) and np.median(rd) < 0, pt_holds=(p2 <= 0.0625) and np.median(pt) > 0))
saw = pd.DataFrame(rows)
print(saw.round(4).to_string())
# how the zero of rat 1 is treated (main reading, peak/trough on the dorsal-most channel)
pt = per_rat(lambda r: RES[r][("asy", "counts", "a", "a")]["peak_trough"][0])
print("peak/trough, main reading, per rat:", np.round(pt, 4))
print("zeros dropped (Wilcoxon; MATLAB signrank): T+ = %.1f, p = %.4f" % m.signrank(pt))
for zm in ["pratt", "zsplit"]:
    print(f"{zm}: T+ = %.1f, exact p = %.4f" % m.signrank_zeros(pt, zm))


**Result: reproduced.** On the dorsal-most channel the rise/decay index is negative in all six rats
(T+ = 0, p = 0.031) under every reading. The peak/trough index is positive in five rats and exactly
0 in rat 1, whose dorsal-most channel has equal median peak and trough durations; the signed-rank
test drops the zero, as Wilcoxon's test does, leaving five positive values, T+ = 15 of 15, p =
0.0625, which meets the criterion's level of 0.0625 (O5). Under three readings rat 1's value is
-0.02, and T+ = 20 of 21, p = 0.0625. The outcome is at the boundary of
the criterion's level whichever way the zero is treated: dropped (T+ = 15, p = 0.0625), ranked and
then dropped (Pratt, T+ = 20, exact p = 0.0625) or split (T+ = 20.5, exact p = 0.0625). With six
rats no treatment gives a smaller p, as at most five values are non-zero and positive.


## fig5-ventral-rise-not-peak

> About 2.8 mm from the dorsal edge, the rising phase is still shorter than the decaying phase, but
> peak and trough durations no longer differ significantly.

The channel nearest 2.8 mm is the ventral-most of each rat (see fig5c-rise-decay-gradient); the
ventral-most variant is therefore the same channel.

In [ ]:
rows = []
for u in UNITS:
    for c in CHR:
        for e in EXT:
            if not all(("asy", u, c, e) in RES[r] for r in m.RATS):
                continue
            idx = {r: int(np.argmin(np.abs(RES[r]["pos"] - 2.8))) for r in m.RATS}
            rd = per_rat(lambda r: RES[r][("asy", u, c, e)]["rise_decay"][idx[r]])
            pt = per_rat(lambda r: RES[r][("asy", u, c, e)]["peak_trough"][idx[r]])
            t1, p1 = m.signrank(rd)
            t2, p2 = m.signrank(pt)
            rows.append(dict(units=u, O3=c, O7=e, rise_decay=" ".join(f"{x:.2f}" for x in rd), T_rd=t1, p_rd=p1,
                             peak_trough=" ".join(f"{x:.2f}" for x in pt), median_pt=np.median(pt), T_pt=t2, p_pt=p2,
                             rd_holds=(p1 <= 0.0625) and np.median(rd) < 0, pt_absent=p2 > 0.0625))
ven = pd.DataFrame(rows)
print("channel used per rat:", {r: int(np.argmin(np.abs(RES[r]['pos'] - 2.8))) + 1 for r in m.RATS}, "(1 = dorsal-most)")
print(ven.round(4).to_string())

**Result: reproduced.** At the channel nearest 2.8 mm (the ventral-most in every rat) the rise/decay
index is negative in all six rats (T+ = 0, p = 0.031 under the main reading; p at most 0.0625 under
every reading), and the peak/trough test, a claim of absence, does not find a difference (main
reading T+ = 4, p = 0.219, median -0.038; p = 0.156 to 0.625 under the other readings).

## fig6b-delay-by-phase

> The conduction delay of theta along the dorsal-ventral axis is shortest for the falling phase
> (5.47 ms/mm) and longest for the peak (12.00 ms/mm), and the two differ significantly.

Per reference cycle of the dorsal-most channel (valid, inside the theta mask), the nearest same
phase on every other channel; robust regression (bisquare) of offset on position; cycles that do
not converge dropped; median over cycles per rat, median over rats.

In [ ]:
rng = np.random.default_rng(2056)
rows = []
for u in UNITS:
    for c in ["b", "c"]:
        for e in EXT:
            if not all(("del", u, c, e) in RES[r] for r in m.RATS):
                continue
            med = {ph: per_rat(lambda r: np.median(RES[r][("del", u, c, e)][ph])) for ph in m.PHASES}
            tp, p = m.signrank(med["peak"] - med["fall"])
            order = [min(m.PHASES, key=lambda ph: med[ph][i]) == "fall" for i in range(6)]
            mm = {ph: np.median(med[ph]) for ph in m.PHASES}
            rows.append(dict(units=u, O3=c, O7=e, fall=mm["fall"], trough=mm["trough"], rise=mm["rise_mid"], peak=mm["peak"],
                             fall_shortest=min(mm, key=mm.get) == "fall", peak_longest=max(mm, key=mm.get) == "peak",
                             T_plus_peak_minus_fall=tp, p=p,
                             n_ref=" ".join(str(RES[r][("del", u, c, e)]["n_ref"]) for r in m.RATS),
                             dropped=" ".join(str(sum(RES[r][("del", u, c, e)][ph + "_dropped"] for ph in m.PHASES)) for r in m.RATS)))
fig6 = pd.DataFrame(rows)
print(fig6.round(3).to_string())
for ph in m.PHASES:
    v = per_rat(lambda r: np.median(RES[r][("del", "counts", "b", "a")][ph]))
    print(f"main (counts, O3 b, O7 a) {ph}: per rat {np.round(v, 2)}, median {np.median(v):.2f}, CI {np.round(m.boot_ci(v, rng), 2)}")
ok = ((fig6.fall - 5.47).abs() <= 3.23) & ((fig6.peak - 12.0).abs() <= 5.48) & fig6.fall_shortest & fig6.peak_longest & (fig6.p <= 0.0625)
print("combinations meeting every part:", int(ok.sum()), "of", len(fig6))
# per rat, main reading: median delay at each phase and the longest phase
D6 = pd.DataFrame({ph: per_rat(lambda r: np.median(RES[r][("del", "counts", "b", "a")][ph])) for ph in m.PHASES}, index=m.RATS)
D6["longest"] = D6[m.PHASES].idxmax(axis=1)
D6["shortest"] = D6[m.PHASES].idxmin(axis=1)
print(D6.round(2).to_string())
print("rats with the rising phase longer than the peak:", int((D6.rise_mid > D6.peak).sum()), "of 6")


**Result: partially reproduced.** Under the main reading the falling-phase delay is 5.86 ms/mm
(within 5.47 +/- 3.23) and the peak delay 11.82 ms/mm (within 12.00 +/- 5.48), the falling phase is
the shortest of the four, and the peak is longer than the falling phase in all six rats (T+ = 21,
p = 0.031). But the peak is not the longest: the rising-phase delay is 12.53 ms/mm, and the rising
phase is the longest of the across-rat medians under all eight readings. Per rat (main reading),
the falling phase is the shortest in all six, and the rising phase is longer than the peak in four
(rats 1, 3, 5 and 6). One part of the criterion fails.


## fig6c-delay-phase-slope

> Conduction delay increases linearly with reference phase between the falling phase and the
> peak, at 0.02 (ms/mm)/deg (R2 = 0.82).

In [ ]:
rows = []
deg = np.array([m.PHASE_DEG[ph] for ph in m.PHASES], float)
for u in UNITS:
    for c in ["b", "c"]:
        for e in EXT:
            if not all(("del", u, c, e) in RES[r] for r in m.RATS):
                continue
            M = np.array([[np.median(RES[r][("del", u, c, e)][ph]) for ph in m.PHASES] for r in m.RATS])
            Mc = M - M.mean(1, keepdims=True)
            X = np.tile(deg, 6)
            slope, icpt, rr, _, _ = stats.linregress(X, Mc.ravel())
            per = [np.polyfit(deg, row, 1)[0] for row in M]
            rows.append(dict(units=u, O3=c, O7=e, pooled_slope=slope, R2=rr**2, median_per_rat_slope=np.median(per),
                             per_rat=" ".join(f"{v:.4f}" for v in per)))
fig6c = pd.DataFrame(rows)
print(fig6c.round(4).to_string())
ok = (((fig6c.pooled_slope >= 0.017) & (fig6c.pooled_slope <= 0.025)) | ((fig6c.median_per_rat_slope >= 0.017) & (fig6c.median_per_rat_slope <= 0.025))) & ((fig6c.R2 - 0.82).abs() <= 0.082)
print("combinations meeting both:", int(ok.sum()), "of", len(fig6c))

**Result: partially reproduced, resting on the unit assumption.** Delay rises with reference
phase: under the main reading the pooled slope is 0.0208 (ms/mm)/deg and the median per-rat slope
0.0194, inside 0.017 to 0.025, but the pooled R2 is 0.668, outside 0.82 +/- 0.082 (0.668 to 0.695
under the `counts` readings). With no amplitude restriction the slope is 0.013 to 0.014, below the
range. The slope holds and the linear fit is weaker than the paper's.

## fig6b-hilbert-between-limits

> Converted to degrees at 8 Hz, the phase-specific shifts range from 15.38 deg/mm (falling phase)
> to 34.56 deg/mm (peak), and the Hilbert-based shift of about 26 deg/mm falls between them.

Delays of fig6b-delay-by-phase times 360/125 deg/ms. Readings of the criterion: (a) the median over
rats of the per-rat medians, converted; (b) the median of the converted per-cycle delays of all
rats pooled, a different statistic, as rats contribute different numbers of cycles. The Hilbert
shift is the main reading of fig2d-phase-shift-rate (O3 a) under the same unit reading.

In [ ]:
rows = []
for u in ["counts", "none"]:
    for c in ["b", "c"]:
        for e in EXT:
            D = {r: RES[r][("del", u, c, e)] for r in m.RATS}
            f = per_rat(lambda r: np.median(D[r]["fall"])) * 360 / 125
            pk = per_rat(lambda r: np.median(D[r]["peak"])) * 360 / 125
            fb = np.median(np.concatenate([D[r]["fall"] for r in m.RATS])) * 360 / 125  # reading b: per cycle, pooled over rats
            pb = np.median(np.concatenate([D[r]["peak"] for r in m.RATS])) * 360 / 125
            # the Hilbert shift is that of fig2d-phase-shift-rate's main reading (O3 a, mean of slopes)
            hil = np.mean(per_rat(lambda r: m.lsq_slope(RES[r]["pos"], RES[r][("hil", u, "a")]["offset"])))
            rows.append(dict(units=u, O3=c, O7=e, fall_a=np.median(f), peak_a=np.median(pk), fall_b=fb, peak_b=pb, hilbert=hil,
                             between_a=np.median(f) < hil < np.median(pk), between_b=fb < hil < pb))
h6 = pd.DataFrame(rows)
print(h6.round(2).to_string())
for rd in ["a", "b"]:
    ok = ((h6[f"fall_{rd}"] - 15.38).abs() <= 9.68) & ((h6[f"peak_{rd}"] - 34.56).abs() <= 16.78) & h6[f"between_{rd}"]
    print(f"reading {rd}: combinations meeting every part:", int(ok.sum()), "of", len(h6))


**Result: reproduced, resting on the unit assumption.** Under the main reading the converted
falling-phase shift is 16.86 deg/mm (within 15.38 +/- 9.68) and the peak shift 34.06 deg/mm (within
34.56 +/- 16.78), and the Hilbert shift of fig2d-phase-shift-rate, 27.32 deg/mm, lies between them;
all four `counts` combinations meet every part under both readings (reading b, the median of the
pooled per-cycle values, gives 18.33 and 32.39 under the main reading). With no amplitude
restriction the peak shift is 23.5 to 27.0 deg/mm, below the Hilbert shift of 27.7, and the claim
fails.


## fig7-asymmetry-delay

> The conduction delay attributable to the change in waveform asymmetry along the axis is 3.56
> ms/mm, significantly greater than zero.

Delay_Asym(j) = (D_fall(0) - D_rise(0)) - (D_fall(j) - D_rise(j)) on each reference cycle, with
the cycle of channel j whose peak is nearest the reference peak. Readings: (a) per cycle robust
slope on position, median over cycles; (b) per channel median, then least-squares slope.

In [ ]:
def asym_b(d, pos):
    return m.lsq_slope(pos, np.median(d["asym_matrix"], axis=0))

rng = np.random.default_rng(2057)
rows = []
for u in UNITS:
    for c in ["b", "c"]:
        for e in EXT:
            if not all(("del", u, c, e) in RES[r] for r in m.RATS):
                continue
            a = per_rat(lambda r: np.median(RES[r][("del", u, c, e)]["asym_cycle"]))
            b = per_rat(lambda r: asym_b(RES[r][("del", u, c, e)], RES[r]["pos"]))
            ta, pa = m.signrank(a)
            tb, pb = m.signrank(b)
            rows.append(dict(units=u, O3=c, O7=e, a_median=np.median(a), a_per_rat=" ".join(f"{v:.2f}" for v in a), T_a=ta, p_a=pa,
                             b_median=np.median(b), b_per_rat=" ".join(f"{v:.2f}" for v in b), T_b=tb, p_b=pb))
f7 = pd.DataFrame(rows)
print(f7.round(3).to_string())
a = per_rat(lambda r: np.median(RES[r][("del", "counts", "b", "a")]["asym_cycle"]))
print("main (a) CI:", np.round(m.boot_ci(a, rng), 2), "; swapped formula (i and j exchanged) gives the negatives:", np.round(-np.median(a), 2))
ok = (((f7.a_median - 3.56).abs() <= 2.33) & (f7.p_a <= 0.0625) & (f7.a_median > 0)) | (((f7.b_median - 3.56).abs() <= 2.33) & (f7.p_b <= 0.0625) & (f7.b_median > 0))
print("combinations meeting every part under a or b:", int(ok.sum()), "of", len(f7))
# variant, not the paper's formula: half of Delay_Asym (a peak displaced by half the change in
# rise-decay difference); the formula of the Methods is the full difference
half = f7[["units", "O3", "O7"]].assign(a_half=f7.a_median / 2, b_half=f7.b_median / 2)
print(half.round(2).to_string())


**Result: not reproduced, resting on the unit assumption.** Under the main reading the
asymmetry-related delay is 6.07 ms/mm (reading a; 6.56 under reading b), outside 3.56 +/- 2.33, and
positive in five of six rats (T+ = 20, p = 0.0625; rat 3 is -1.64). With the dorsal-most channel's
epochs it is 6.70 to 7.02, positive in all six (p = 0.031). Only with no amplitude restriction do
two combinations come within tolerance (5.66 and 5.69, p = 0.031). Exchanging i and j in the
formula gives the negatives, which have the wrong sign. The threshold sweep gives 6.0
to 7.6 ms/mm at every threshold that can be computed. Half of Delay_Asym (a variant, not the
Methods' formula, which is the full difference) gives 2.8 to 3.5 ms/mm, within tolerance; it is
shown because a peak displaced by a change of rise and decay durations at a fixed period moves by
half the change in their difference, but the Methods define the delay as the full difference, and
the outcome is that of the Methods' formula.


## fig7-residual-delay

> Subtracting the asymmetry-related delay (3.56 ms/mm) from the total observed conduction delay
> (12.98 ms/mm) leaves a residual delay of 9.61 ms/mm.

Totals: (a) the peak delay of fig6b-delay-by-phase; (b) per-channel median peak offset regressed on
distance; (c) the Hilbert shift at 8 Hz (deg/mm times 125/360). Residuals: per cycle under (a), per
channel under (b), and (c) minus the asymmetry delay of reading b.

In [ ]:
rows = []
for u in UNITS:
    for c in ["b", "c"]:
        for e in EXT:
            if not all(("del", u, c, e) in RES[r] for r in m.RATS):
                continue
            D = {r: RES[r][("del", u, c, e)] for r in m.RATS}
            pos = {r: RES[r]["pos"] for r in m.RATS}
            ta = per_rat(lambda r: np.median(D[r]["peak"]))
            ra = per_rat(lambda r: np.median(D[r]["resid_cycle"][D[r]["resid_ok"]]))
            tb = per_rat(lambda r: m.lsq_slope(pos[r], np.median(D[r]["peak_matrix"], axis=0)))
            rb = per_rat(lambda r: m.lsq_slope(pos[r], np.median(D[r]["peak_matrix"] - D[r]["asym_matrix"], axis=0)))
            hc = "a"
            tc = per_rat(lambda r: m.lsq_slope(pos[r], RES[r][("hil", u, hc)]["offset"])) * 125 / 360 if not any(np.isnan(RES[r][("hil", u, hc)]["offset"]).any() for r in m.RATS) else np.full(6, np.nan)
            rc = tc - per_rat(lambda r: asym_b(D[r], pos[r]))
            for name, t, res_ in [("a", ta, ra), ("b", tb, rb), ("c", tc, rc)]:
                if np.isnan(t).any():
                    continue
                tp, p = m.signrank(t - res_)
                rows.append(dict(units=u, O3=c, O7=e, total_reading=name, total=np.median(t), residual=np.median(res_),
                                 residual_below_total_all=bool((res_ < t).all()), T_plus=tp, p=p,
                                 total_per_rat=" ".join(f"{v:.1f}" for v in t), residual_per_rat=" ".join(f"{v:.1f}" for v in res_)))
f7r = pd.DataFrame(rows)
print(f7r.round(3).to_string())
ok = ((f7r.total - 12.98).abs() <= 5.23) & ((f7r.residual - 9.61).abs() <= 2.62) & f7r.residual_below_total_all
print("combinations meeting every part:", int(ok.sum()), "of", len(f7r))
print(f7r[ok][["units", "O3", "O7", "total_reading", "total", "residual"]].round(2).to_string())
# variant: residual with half of Delay_Asym removed per cycle (total reading a)
rows = []
for u in UNITS:
    for c in ["b", "c"]:
        for e in EXT:
            if not all(("del", u, c, e) in RES[r] for r in m.RATS):
                continue
            D = {r: RES[r][("del", u, c, e)] for r in m.RATS}
            rows.append(dict(units=u, O3=c, O7=e, total=np.median(per_rat(lambda r: np.median(D[r]["peak"]))),
                             residual_half=np.median(per_rat(lambda r: np.median(D[r]["resid_half_cycle"])))))
print(pd.DataFrame(rows).round(2).to_string())


**Result: not reproduced.** The total delay is within 12.98 +/- 5.23 ms/mm under every reading (8.7
to 11.8 ms/mm), but because the asymmetry-related delay is about twice the paper's, the residual is
2.0 to 5.2 ms/mm, below 9.61 +/- 2.62 under all 24 readings (5.24 under the main reading). The
residual is below the total in every rat under 18 of the 24 readings. With half of Delay_Asym removed
(the variant of fig7-asymmetry-delay) the residual is 7.86 to 9.20 ms/mm under the `counts`
readings, within 9.61 +/- 2.62 (6.99 to 12.23), and 6.80 to 7.49 with no amplitude restriction,
two of which (6.80 and 6.81) are below it; the outcome is that of the Methods' formula.


## fig8c-relative-delay-flat, fig8d-absolute-delay-slope, fig8-correlation-strength

Per reference cycle, the asymmetry-corrected peak delay (the per-cycle residual of
fig7-residual-delay, reading a), in ms/mm; relative delay is that delay over the cycle's period
in percent; frequency is the inverse of the period. Period: the dorsal-most channel's peak to next
peak (the claims file's reading) or previous peak to peak (shown, as the criterion asks). A third
variant, half the interval from the previous to the next peak (`centred`), is this attempt's own:
it does not use the reference peak's time, which enters the delay as well, and so shows how much of
a relation comes from timing noise shared by the delay and the period. A fourth variant, `hilbert`, takes the frequency
from the dorsal-most channel's Hilbert phase, averaged over a fixed 250 ms window centred on the
reference peak, so it does not use the extrema's times at all. Per rat, the least-squares slope on
frequency and the Pearson r. Absolute delay reading (b) multiplies the delay by the rat's span.

In [ ]:
def fig8_rat(d, pos, period_key, span):
    if period_key == "centred":
        allper = (d["period"] + d["prev_period"]) / 2
    elif period_key == "hilbert":
        allper = 1000.0 / d["hilbert_freq"]
    else:
        allper = d[period_key]
    ok = d["resid_ok"] & np.isfinite(allper)
    absd = d["resid_cycle"][ok] * (pos[-1] if span else 1.0)
    per = allper[ok]
    freq = 1000.0 / per
    rel = d["resid_cycle"][ok] / per * 100
    return dict(abs_slope=np.polyfit(freq, absd, 1)[0], rel_slope=np.polyfit(freq, rel, 1)[0],
                abs_r=stats.pearsonr(freq, absd)[0], rel_r=stats.pearsonr(freq, rel)[0],
                abs_rho=stats.spearmanr(freq, absd)[0], rel_rho=stats.spearmanr(freq, rel)[0], n=int(ok.sum()))

rng = np.random.default_rng(2058)
F8 = []
for u in UNITS:
    for c in ["b", "c"]:
        for e in EXT:
            if not all(("del", u, c, e) in RES[r] for r in m.RATS):
                continue
            for pk in ["period", "prev_period", "centred", "hilbert"]:
                for span in [False, True]:
                    res_ = [fig8_rat(RES[r][("del", u, c, e)], RES[r]["pos"], pk, span) for r in m.RATS]
                    row = dict(units=u, O3=c, O7=e, period=pk, span=span)
                    for k in ["abs_slope", "rel_slope", "abs_r", "rel_r", "abs_rho", "rel_rho"]:
                        row[k] = np.array([q[k] for q in res_])
                    row["n"] = [q["n"] for q in res_]
                    F8.append(row)
print(f"{len(F8)} combinations")

In [ ]:
rows = []
for row in F8:
    if row["span"]:
        continue
    s = row["rel_slope"]
    tp, p = m.signrank(s)
    rows.append(dict(units=row["units"], O3=row["O3"], O7=row["O7"], period=row["period"], median_slope=np.median(s),
                     per_rat=" ".join(f"{v:.2f}" for v in s), T_plus=tp, p=p, absent=p > 0.0625))
f8c = pd.DataFrame(rows)
print(f8c.round(4).to_string())
main = [r for r in F8 if (r["units"], r["O3"], r["O7"], r["period"], r["span"]) == ("counts", "b", "a", "period", False)][0]
print("main: median relative slope", np.round(np.median(main["rel_slope"]), 3), "%/Hz, CI", np.round(m.boot_ci(main["rel_slope"], rng), 3), "; cycles per rat", main["n"])

**Result: not reproduced, resting on the period reading.** With the period from the reference peak
to the next peak (the claims file's reading) the relative delay rises with frequency in all six
rats (median slope 2.16 %/Hz, 95% CI 1.61 to 2.52, T+ = 21, p = 0.031), where the paper finds no
relation; with the period from the previous peak it falls in all six (median -1.28 %/Hz, p =
0.031). Both of the criterion's readings find a relation. Two variants of this attempt, which do
not tie the period to the reference peak's time, find none under most `counts` readings: the
centred period (p = 0.44 to 0.69) and the Hilbert frequency (p = 0.156 and 0.5625 under three of
the four `counts` combinations; 0.031 under O3 c with O7 a). The sign flip between the two
readings shows that the relation they find comes largely from timing shared by the per-cycle delay
and the period, so this outcome says that the paper's result is not found with the claims file's
definition of the period; it is not evidence against the paper's mechanism.


In [ ]:
rows = []
for row in F8:
    s = row["abs_slope"]
    tp, p = m.signrank(s)
    rows.append(dict(units=row["units"], O3=row["O3"], O7=row["O7"], period=row["period"], span=row["span"], median_slope=np.median(s),
                     per_rat=" ".join(f"{v:.2f}" for v in s), T_plus=tp, p=p))
f8d = pd.DataFrame(rows)
print(f8d.round(4).to_string())
ok = ((f8d.median_slope + 3.12).abs() <= 0.41) & (f8d.p <= 0.0625) & (f8d.T_plus == 0)
print("combinations meeting every part:", int(ok.sum()), "of", len(f8d))
print("main: median absolute slope", np.round(np.median(main["abs_slope"]), 3), "ms/mm per Hz, CI", np.round(m.boot_ci(main["abs_slope"], rng), 3))

In [ ]:
# per-rat sensitivity to the timing of the reference peak (main unit, O3 and O7 readings): slope of
# the per-cycle delay on frequency, with the delay measured at the peak (corrected, the claims'
# delay; or uncorrected) or at the trough, whose time is not the reference peak's
d = {r: RES[r][("del", "counts", "b", "a")] for r in m.RATS}
rows = []
for dname, getd in [("peak, corrected (claims)", lambda q: (q["resid_cycle"], q["resid_ok"])),
                    ("peak, uncorrected", lambda q: (q["peak_all"], q["peak_ok"])),
                    ("trough, uncorrected", lambda q: (q["trough_all"], q["trough_ok"]))]:
    for pk in ["period", "prev_period", "centred", "hilbert"]:
        sl = []
        for r in m.RATS:
            q = d[r]
            dl, ok = getd(q)
            per = {"period": q["period"], "prev_period": q["prev_period"], "centred": (q["period"] + q["prev_period"]) / 2,
                   "hilbert": 1000.0 / q["hilbert_freq"]}[pk]
            ok = ok & np.isfinite(per) & np.isfinite(dl)
            sl.append(np.polyfit(1000.0 / per[ok], dl[ok], 1)[0])
        sl = np.array(sl)
        tp, p = m.signrank(sl)
        rows.append(dict(delay=dname, period=pk, per_rat=" ".join(f"{v:.2f}" for v in sl), median=np.median(sl), T_plus=tp, p=p))
print("absolute delay (ms/mm) on frequency (Hz), per rat")
print(pd.DataFrame(rows).round(3).to_string())


**Result: not reproduced, resting on the period reading.** With the period to the next peak the
absolute delay rises with frequency in all six rats (median slope +2.06 ms/mm per Hz, 95% CI 1.28
to 2.76, p = 0.031; +5.16 ms per Hz over the span), opposite in sign to the paper's -3.12 ms/Hz.
With the period from the previous peak (shown, as the criterion of fig8c-relative-delay-flat asks)
the slope is negative in all rats (-2.36 under the main reading, outside the tolerance), and with no
amplitude restriction it is -3.13 and -3.12 ms/mm per Hz under two combinations, within tolerance.

The per-rat table above shows where the sign comes from. The uncorrected peak delay falls with
frequency under the next-peak period (median -2.07, all six rats) and the trough delay, whose time
is not the reference peak's, falls under the next-peak, centred and Hilbert frequencies (-0.74,
-1.15, -1.85). The asymmetry correction subtracts the reference cycle's decay minus rise, which
shares the reference peak and next trough with the period, and turns the corrected slope positive.
With the Hilbert frequency, which uses no extrema times, the corrected delay falls with frequency
(-0.89 ms/mm per Hz under the main reading, p = 0.094; -0.40 to -1.35 under the eight combinations,
p = 0.031 under five), and with the delay over the span and no amplitude restriction it gives -3.00
and -2.93 ms per Hz, within tolerance. The direction of the paper's finding appears when the period
does not share timing with the delay; its size per mm does not. This variant is this attempt's,
not the claims file's, and does not decide the outcome.


In [ ]:
rows = []
for row in F8:
    if row["span"]:
        continue
    for name, fa, fr in [("|r|", np.abs(row["abs_r"]), np.abs(row["rel_r"])), ("r2", row["abs_r"] ** 2, row["rel_r"] ** 2),
                         ("|rho| (variant)", np.abs(row["abs_rho"]), np.abs(row["rel_rho"]))]:
        tp, p = m.signrank(fa - fr)
        rows.append(dict(units=row["units"], O3=row["O3"], O7=row["O7"], period=row["period"], measure=name,
                         relative=np.median(fr), absolute=np.median(fa), T_plus_abs_minus_rel=tp, p=p,
                         rel_per_rat=" ".join(f"{v:.3f}" for v in fr), abs_per_rat=" ".join(f"{v:.3f}" for v in fa)))
f8r = pd.DataFrame(rows)
print(f8r.round(4).to_string())
ok = ((f8r.relative - 0.02).abs() <= 0.02) & ((f8r.absolute - 0.14).abs() <= 0.05) & (f8r.p <= 0.0625) & (f8r.absolute > f8r.relative) & (f8r.measure != "|rho| (variant)")
print("combinations meeting every part:", int(ok.sum()), "of", int((f8r.measure != '|rho| (variant)').sum()))

**Result: not reproduced, resting on the period reading.** Under the main reading the median |r| is
0.260 for the relative delay and 0.207 for the absolute delay, the relative delay being the more
strongly related to frequency (T+ = 0, p = 0.031), the reverse of the paper's finding (0.02
against 0.14). No combination of |r| or r2 puts both medians within tolerance with the test met
(0 of 64, the centred and Hilbert variants included). With the centred period the relative
delay's |r| is 0.025 to 0.057 and the absolute delay's 0.050 to 0.087 under the `counts`
readings, the paper's order; with the Hilbert frequency 0.029 to 0.060 against 0.043 to 0.070. The
absolute delay's |r| is well below the paper's 0.14 under both variants. Under the `counts`
readings the test finds the absolute delay the more strongly related under one Hilbert combination
(O3 c, O7 b; p = 0.031 with |r| and with r2) and, with r2 only, under one centred combination (O3
c, O7 b; T+ = 20, p = 0.0625), where the absolute delay's median r2 is 0.005; with |r| it finds no
centred one (p = 0.156 to 0.438).


## fig8-coupled-oscillators

> The residual phase shifts behave as weakly coupled oscillators predict and not as a fixed-delay
> mechanism.

Reproduced when fig8c-relative-delay-flat and fig8d-absolute-delay-slope are both reproduced; not
reproduced when the relative delay rises significantly with frequency and the absolute delay does
not change (the fixed-delay pattern); partially reproduced otherwise.

In [ ]:
s_rel = main["rel_slope"]
s_abs = main["abs_slope"]
t1, p1 = m.signrank(s_rel)
t2, p2 = m.signrank(s_abs)
print(f"relative delay on frequency: median slope {np.median(s_rel):.3f} %/Hz, T+ {t1:.0f}, p {p1:.4f}")
print(f"absolute delay on frequency: median slope {np.median(s_abs):.3f} ms/mm per Hz, T+ {t2:.0f}, p {p2:.4f}")
print("fixed-delay pattern (relative rises significantly, absolute unchanged):", (p1 <= 0.0625 and np.median(s_rel) > 0) and p2 > 0.0625)
print("coupled-oscillator pattern (relative unchanged, absolute falls):", p1 > 0.0625 and (p2 <= 0.0625 and np.median(s_abs) < 0))

**Result: partially reproduced.** Under the claims file's reading neither fig8c-relative-delay-flat
nor fig8d-absolute-delay-slope is reproduced: the relative delay and the absolute delay both rise
with frequency (median slopes 2.16 %/Hz and 2.06 ms/mm per Hz, both p = 0.031). This is not the
fixed-delay pattern either, as the absolute delay changes. The criterion calls this case partially
reproduced, though neither of the coupled-oscillator predictions holds. With the Hilbert frequency (a variant
that does not share timing with the delay; see fig8d-absolute-delay-slope) the relative delay does
not change under the main reading (p = 0.156) and the absolute delay falls (median -0.89 ms/mm per
Hz, p = 0.094), the direction the coupled-oscillator account predicts, without the test finding
the fall under the main reading.


## Blocked claims

- fig3-along-not-across-layers: the laminar array (up to six stereotrodes per shank, Figure 3A) is
  not in the data set; each rat's file holds 5 or 6 channels in one dorsal-ventral order, with no
  geometry, so the along- and across-layer displacements cannot be formed. Data missing.
- fig4d-mua-phase-shift and fig4d-mua-matches-lfp: the multi-unit band is 600-3000 Hz (Methods,
  Analysis of multiunit spiking activity); the files are sampled at 500 Hz, with nothing above 250
  Hz, and hold one recording per rat, the LFP of the key recording, not the later layer II
  recordings. Data missing.

In [ ]:
print(f"total run time {time.time() - T0:.0f} s, peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

| Claim | Outcome |
|---|---|
| fig2d-phase-shift-rate | reproduced |
| fig2d-linear-fit | reproduced |
| fig2a-all-rats | reproduced |
| fig2e-phase-locking-high | reproduced (assumed) |
| fig2e-phase-locking-decrease | reproduced (assumed) |
| fig2c-coherence | reproduced |
| fig2f-coherence-decrease | not reproduced |
| fig2g-power-no-change | reproduced |
| fig3-along-not-across-layers | blocked, data missing |
| fig4d-mua-phase-shift | blocked, data missing |
| fig4d-mua-matches-lfp | blocked, data missing |
| fig5c-rise-decay-gradient | partially reproduced |
| fig5c-peak-trough-gradient | reproduced |
| fig5-dorsal-sawtooth | reproduced |
| fig5-ventral-rise-not-peak | reproduced |
| fig6b-delay-by-phase | partially reproduced |
| fig6c-delay-phase-slope | partially reproduced (assumed) |
| fig6b-hilbert-between-limits | reproduced (assumed) |
| fig7-asymmetry-delay | not reproduced (assumed) |
| fig7-residual-delay | not reproduced |
| fig8c-relative-delay-flat | not reproduced (assumed: period reading) |
| fig8d-absolute-delay-slope | not reproduced (assumed: period reading) |
| fig8-correlation-strength | not reproduced (assumed: period reading) |
| fig8-coupled-oscillators | partially reproduced (assumed: period reading) |

"(assumed)": the outcome rests on the unit reading `counts`, which applies the paper's 400 µV
threshold on one scale for all rats (400 µV if the integer files are in µV); the outcome under the
other readings and thresholds is in the claim's section and the threshold sweep. "(assumed: period
reading)": the outcome rests on the claims file's definition of a cycle's period, which shares
timing with the per-cycle delay; variants that do not share it are in the Figure 8 sections.

How the analysis differs from the paper's: electrode positions are the nominal 0.566 mm spacing,
as the data set gives none; every channel in a file is used, and the data set's author proof
marks as many electrodes at the key recording as the files hold channels; the units of the files are not stated, and the 400 µV threshold is
applied on the scale of the integer files, with rats 2 and 3 divided by 0.195 (an assumption; the
threshold is 400 µV only if the integer files are in µV); the paper's
code was not available, so the waveform extrema, the robust regression and the matching of cycles
across channels are this attempt's implementation of the Methods' text.

What was found about the data set: it holds the LFP of the six key recordings at 500 Hz, not the
30 kHz broadband, the multi-unit recordings, or the laminar array's sites; it states no unit, and
two of its six files (rats 2 and 3) are non-integer and on a scale about five times smaller than
the other four, so the paper's 400 µV threshold selects no theta epoch in them as stored; it gives
no electrode positions, and marks the channels used only in a figure of its author proof. The per-cycle relation of conduction delay to
frequency (Figure 8) changes sign with the definition of the cycle's period, a sign that it is
dominated by timing noise shared by the two measures; with a frequency taken from the Hilbert
phase the median slope of the absolute delay on frequency is negative under every reading, the
paper's direction, though smaller per mm, but the signed-rank test finds the fall in only one of
the four `counts` readings (p = 0.094 under the main reading) and in all four with no amplitude
restriction.
Variants shown but not deciding an outcome: the natural log for the Figure 5C indices, which
brings fig5c-rise-decay-gradient within tolerance (asked of a reader), and half of the
asymmetry-related delay, which is not the Methods' formula and brings fig7-asymmetry-delay within
tolerance under every reading and fig7-residual-delay under the `counts` readings and two of the
four readings with no amplitude restriction.